In [28]:
import torch,os,time
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

In [2]:
model_id = "openai/whisper-large-v3"

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

"device : ",device,"torch_dtype : ",torch_dtype

('device : ', 'cuda:0', 'torch_dtype : ', torch.float16)

In [3]:
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(128, 1280, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(1280, 1280, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 1280)
      (layers): ModuleList(
        (0-31): 32 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=1280, out_features=1280, bias=False)
            (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
            (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
            (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1280, out_features=5120, bias=True)
          (fc2): Linear(in_features=5120, out_features=1280, bias=Tr

In [4]:
processor = AutoProcessor.from_pretrained(model_id)

In [5]:
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=30,
    dtype=torch_dtype,
    device=device,
)


[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


In [6]:
generate_kwargs = {
    "return_timestamps": True,
}

In [7]:
old_time=time.time()
result = pipe("kesit.mp3" ,return_timestamps=True,generate_kwargs={"language": "english"})

print("processes_time : ",time.time()-old_time)

[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

processes_time :  5.366365432739258


In [ ]:
8*4

In [8]:
result["chunks"]

[{'timestamp': (0.0, 26.0),
  'text': " It's been a long day without you, my friend And I'll tell you all about it when I see you again. We've come a long way"},
 {'timestamp': (26.0, 27.86), 'text': ' from where we'},
 {'timestamp': (27.86, 30.0), 'text': " began. Oh, I'll"}]

In [4]:
import librosa
import soundfile as sf
import numpy as np

In [6]:
 # Ses dosyasını yükle
y, sr = librosa.load('dene.wav', sr=16000)

# Örnek indislerini hesapla
başlangıç_sample = int(0 * sr)
bitiş_sample = int((20 + 1000) * sr)

# Seçilen bölümü kes
kesilen_ses = y[başlangıç_sample:bitiş_sample]

C:\Users\yasin\AppData\Local\Temp\ipykernel_19648\286266927.py:2: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load('dene.wav', sr=16000)
C:\Users\yasin\AppData\Roaming\Python\Python313\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


In [8]:
kesilen_ses

array([ 0.        ,  0.        ,  0.        , ..., -0.03944165,
       -0.03621333, -0.02734058], shape=(16320000,), dtype=float32)

In [10]:
import torch,re
import time
from tqdm import tqdm
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
import librosa 

In [11]:
# Kaynaklarda kullanılan temel ayarlar [4]
model_id = "openai/whisper-large-v3"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

In [12]:
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
).to(device)

processor = AutoProcessor.from_pretrained(model_id)


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [13]:
input_features = processor(kesilen_ses, sampling_rate=sr, return_tensors="pt").input_features
input_features = input_features.to(device).to(torch_dtype)

In [14]:
input_features

tensor([[[-0.5801, -0.5801, -0.5801,  ...,  0.4290,  0.4951,  0.5356],
         [-0.5801, -0.5801, -0.5801,  ...,  0.5264,  0.5928,  0.6333],
         [-0.5801, -0.5801, -0.5801,  ...,  0.7759,  0.7969,  0.7632],
         ...,
         [-0.5801, -0.5801, -0.5801,  ..., -0.3521, -0.5801, -0.5801],
         [-0.5801, -0.5801, -0.5801,  ..., -0.4199, -0.5801, -0.5801],
         [-0.5801, -0.5801, -0.5801,  ..., -0.5801, -0.5801, -0.5801]]],
       device='cuda:0', dtype=torch.float16)

In [15]:
predicted_ids = model.generate(input_features)

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its para

In [16]:
transcription = processor.batch_decode(*predicted_ids, skip_special_tokens=True,decode_with_timestamps=True)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [17]:
transcription

[" It's been a long day without you, my friend And I'll tell you all about it when I see you again We've come a long way from where we began"]

In [13]:

def ses_kes_librosa(giriş_dosyası, çıkış_dosyası, başlangıç=0, süre=30):
    """
    Librosa kullanarak ses dosyasını keser.
    
    Parametreler:
    - giriş_dosyası: Orijinal ses dosyası
    - çıkış_dosyası: Çıktı dosyası
    - başlangıç: Başlangıç zamanı (saniye)
    - süre: Kesim süresi (saniye)
    """
    # Ses dosyasını yükle
    y, sr = librosa.load(giriş_dosyası, sr=None)
    
    # Örnek indislerini hesapla
    başlangıç_sample = int(başlangıç * sr)
    bitiş_sample = int((başlangıç + süre) * sr)
    
    # Seçilen bölümü kes
    kesilen_ses = y[başlangıç_sample:bitiş_sample]
    
    # Sonucu kaydet
    sf.write(çıkış_dosyası, kesilen_ses, sr)
    print(f"✓ Ses başarıyla kesildi: {çıkış_dosyası}")



In [15]:
# Kullanım
ses_kes_librosa('seeyou.mp3', 'kesit.mp3', başlangıç=0, süre=30)

✓ Ses başarıyla kesildi: kesit.mp3


In [7]:
import subprocess

In [8]:
def ses_kes(giriş_dosyası, çıkış_dosyası, başlangıç=0, süre=30):
    """
    Ses dosyasından belirli bir bölümü keser.
    
    Parametreler:
    - giriş_dosyası: Orijinal ses dosyasının yolu
    - çıkış_dosyası: Kesilen dosyanın kaydedileceği yer
    - başlangıç: Başlangıç zamanı (saniye, varsayılan: 0)
    - süre: Kesilecek süre (saniye, varsayılan: 30)
    """
    komut = [
        'ffmpeg',
        '-i', giriş_dosyası,
        '-ss', str(başlangıç),
        '-t', str(süre),
        '-c', 'copy',
        çıkış_dosyası
    ]
    
    subprocess.run(komut, check=True)
    print(f"✓ Ses başarıyla kesildi: {çıkış_dosyası}")



In [11]:
ses_kes('dene.mp3', 'kesilen_müzik.mp3', başlangıç=200, süre=230)

✓ Ses başarıyla kesildi: kesilen_müzik.mp3
